# EPIC Clarity Measurement Hydration

This notebook hydrates the OMOP MEASUREMENT table from EPIC Clarity lab results and vital signs.

## Source Tables
- `_exponent._bronze_epic_clarity_*.dbo_ORDER_RESULTS` - Lab/result findings
- `_exponent._bronze_epic_clarity_*.dbo_V_EHI_FLO_MEAS_EDITED` - Vital signs and flowsheet measurements

## OMOP Fields Populated
- measurement_id (surrogate key)
- measurement_source_value
- measurement_concept_id (mapped from test codes)
- measurement_date
- value_source_value
- unit_source_value
- unit_concept_id
- visit_occurrence_id (if available)

In [ ]:
source = 'epic_clarity'

In [ ]:
silver_measurement_df = spark.sql(f'''
SELECT
    CONCAT_WS(CHR(31), 'epic_clarity', 'ORDER_RESULTS', 'ORDER_PROC_ID', ore.ORDER_PROC_ID) AS measurement_source_value,
    ore.COMPONENT_ID_NAME AS measurement_code_source_value,
    ore.RESULT_DATE AS measurement_date,
    ore.ORD_VALUE AS value_source_value,
    ore.REFERENCE_UNIT AS unit_source_value,
    CONCAT_WS(CHR(31), 'epic_clarity', 'PAT_ENC', 'PAT_ENC_CSN_ID', ore.PAT_ENC_CSN_ID) AS visit_occurrence_source_value,
    CURRENT_TIMESTAMP() AS updated_tsp
FROM _exponent._bronze_epic_clarity.order_results ore
WHERE ore.ORDER_PROC_ID IS NOT NULL

UNION ALL

SELECT
    CONCAT_WS(CHR(31), 'epic_clarity', 'V_EHI_FLO_MEAS_EDITED', 'FSD_ID', vef.FSD_ID) AS measurement_source_value,
    vef.FLO_MEAS_ID_DISP_NAME AS measurement_code_source_value,
    CAST(vef.RECORDED_TIME AS DATE) AS measurement_date,
    vef.MEAS_VALUE_EXTERNAL AS value_source_value,
    vef.UNITS AS unit_source_value,
    NULL AS visit_occurrence_source_value,
    CURRENT_TIMESTAMP() AS updated_tsp
FROM _exponent._bronze_epic_clarity.ip_flwsht_meas vef
WHERE vef.FSD_ID IS NOT NULL
''')

display(silver_measurement_df)
silver_measurement_df.createOrReplaceTempView("silver_measurement")

In [ ]:
%sql
MERGE INTO _exponent.omop_silver.measurement AS target
USING silver_measurement AS source
ON target.measurement_source_value = source.measurement_source_value

WHEN MATCHED AND NOT (
    target.measurement_code_source_value <=> source.measurement_code_source_value
    AND target.measurement_date <=> source.measurement_date
    AND target.value_source_value <=> source.value_source_value
    AND target.unit_source_value <=> source.unit_source_value
    AND target.visit_occurrence_source_value <=> source.visit_occurrence_source_value
)
THEN UPDATE SET
    target.measurement_code_source_value = source.measurement_code_source_value,
    target.measurement_date = source.measurement_date,
    target.value_source_value = source.value_source_value,
    target.unit_source_value = source.unit_source_value,
    target.visit_occurrence_source_value = source.visit_occurrence_source_value,
    target.updated_tsp = source.updated_tsp

WHEN NOT MATCHED THEN INSERT (
    measurement_source_value,
    measurement_code_source_value,
    measurement_date,
    value_source_value,
    unit_source_value,
    visit_occurrence_source_value,
    updated_tsp
)
VALUES (
    source.measurement_source_value,
    source.measurement_code_source_value,
    source.measurement_date,
    source.value_source_value,
    source.unit_source_value,
    source.visit_occurrence_source_value,
    source.updated_tsp
)

In [ ]:
%sql
INSERT INTO _exponent.omop_mapping.source_to_measurement (
    source_system,
    measurement_source_value,
    active_flag,
    created_tsp,
    last_mod_tsp,
    merge_id,
    merge_reason
)
SELECT
    s.source_system,
    s.measurement_source_value,
    TRUE AS active_flag,
    CURRENT_TIMESTAMP() AS created_tsp,
    COALESCE(s.updated_tsp, CURRENT_TIMESTAMP()) AS last_mod_tsp,
    NULL AS merge_id,
    NULL AS merge_reason
FROM (
    SELECT DISTINCT 'epic_clarity' AS source_system, measurement_source_value, updated_tsp
    FROM _exponent.omop_silver.measurement
    WHERE measurement_source_value IS NOT NULL
) s
LEFT ANTI JOIN _exponent.omop_mapping.source_to_measurement x
    ON s.measurement_source_value = x.measurement_source_value
    AND x.source_system = 'epic_clarity'

In [ ]:
gold_measurement_df = spark.sql("""
SELECT
    m.measurement_id,
    s.measurement_date,
    s.value_source_value,
    s.unit_source_value,
    COALESCE(mcm.concept_id, 0) AS measurement_concept_id,
    mcm.unit_concept_id,
    mv.visit_occurrence_id,
    s.updated_tsp
FROM _exponent.omop_silver.measurement s
INNER JOIN _exponent.omop_mapping.source_to_measurement m
    ON s.measurement_source_value = m.measurement_source_value
    AND m.source_system = 'epic_clarity'
    AND m.active_flag = TRUE
LEFT JOIN _exponent.omop_mapping.domain_source_to_concept mcm
    ON mcm.source_id = s.measurement_code_source_value
    AND mcm.domain_id = 'Measurement'
    AND mcm.source_system = 'epic_clarity'
LEFT JOIN _exponent.omop_mapping.source_to_visit_occurrence mv
    ON s.visit_occurrence_source_value = mv.visit_occurrence_source_value
    AND mv.source_system = 'epic_clarity'
    AND mv.active_flag = TRUE
""")

display(gold_measurement_df)
gold_measurement_df.createOrReplaceTempView("gold_measurement")

In [ ]:
%sql
MERGE INTO _exponent.omop.measurement AS target
USING gold_measurement AS source
ON target.measurement_id = source.measurement_id

WHEN MATCHED AND NOT (
    target.measurement_concept_id <=> source.measurement_concept_id
    AND target.measurement_date <=> source.measurement_date
    AND target.value_source_value <=> source.value_source_value
    AND target.unit_source_value <=> source.unit_source_value
    AND target.unit_concept_id <=> source.unit_concept_id
    AND target.visit_occurrence_id <=> source.visit_occurrence_id
)
THEN UPDATE SET
    target.measurement_concept_id = source.measurement_concept_id,
    target.measurement_date = source.measurement_date,
    target.value_source_value = source.value_source_value,
    target.unit_source_value = source.unit_source_value,
    target.unit_concept_id = source.unit_concept_id,
    target.visit_occurrence_id = source.visit_occurrence_id

WHEN NOT MATCHED THEN INSERT (
    measurement_id,
    measurement_concept_id,
    measurement_date,
    value_source_value,
    unit_source_value,
    unit_concept_id,
    visit_occurrence_id
)
VALUES (
    source.measurement_id,
    source.measurement_concept_id,
    source.measurement_date,
    source.value_source_value,
    source.unit_source_value,
    source.unit_concept_id,
    source.visit_occurrence_id
)